# YOLO Auto-Training Colab 测试

本 Notebook 用于在 Google Colab 上测试 yolo-auto-trainning 项目

**注意**: 需要在运行时选择 GPU (Runtime → Change runtime type → T4 GPU)

## Step 1: 环境检查

In [ ]:
# GPU 检查
import subprocess
result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print(result.stdout)

# PyTorch + CUDA 检查
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: 安装依赖

In [ ]:
# 安装核心依赖
!pip install ultralytics>=8.0.0 ray[tune]>=2.0.0 -q
!pip install fastapi uvicorn pydantic pydantic-settings -q

# 验证安装
import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")

## Step 3: 从 GitHub 克隆项目

In [ ]:
# 克隆 yolo-auto-trainning 项目
!git clone https://github.com/muyi-dev/yolo-auto-trainning.git /content/yolo-auto-trainning

print("项目已克隆到 /content/yolo-auto-trainning")
!ls -la /content/yolo-auto-trainning/

## Step 4: 训练测试 (核心功能)

In [ ]:
from ultralytics import YOLO
import torch

# 选择设备
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {device}")

# 加载模型 (使用 nano 模型快速测试)
model = YOLO("yolo11n.pt")

print("\n" + "="*50)
print("开始训练测试...")
print("="*50)

# 训练配置 - 使用内置 coco128 数据集快速验证
results = model.train(
    data="coco128.yaml",  # Ultralytics 内置数据集
    epochs=3,              # 快速验证用少量 epoch
    imgsz=320,            # 小尺寸加速
    batch=16,             # 根据 GPU 内存调整
    device=device,
    project="/content/runs",
    exist_ok=True,
    verbose=True
)

print("\n" + "="*50)
print("训练完成!")
print("="*50)

## Step 5: 查看训练结果

In [ ]:
# 查看训练输出目录
import os
print("训练输出目录内容:")
for root, dirs, files in os.walk("/content/runs"):
    level = root.replace("/content/runs", '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # 只显示前5个文件
        print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... 还有 {len(files)-5} 个文件')

## Step 6: 模型导出测试

In [ ]:
print("\n" + "="*50)
print("测试模型导出...")
print("="*50)

# 获取最佳模型路径
best_model_path = "/content/runs/train/weights/best.pt"
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    
    # 导出为 ONNX
    onnx_path = model.export(format="onnx", imgsz=320)
    print(f"ONNX 导出成功: {onnx_path}")
    
    # 检查导出文件
    import os
    if os.path.exists(onnx_path):
        size_mb = os.path.getsize(onnx_path) / (1024*1024)
        print(f"ONNX 文件大小: {size_mb:.2f} MB")
else:
    print("最佳模型文件不存在，跳过导出测试")

## Step 7: 验证项目结构

In [ ]:
# 验证项目结构
import os
import sys

print("项目结构验证:")
print("="*50)

# 重新设置路径 (在克隆之后)
sys.path.insert(0, '/content/yolo-auto-trainning')
sys.path.insert(0, '/content/yolo-auto-trainning/training-api/src')
sys.path.insert(0, '/content/yolo-auto-trainning/training-api')

# 检查关键目录
dirs_to_check = [
    '/content/yolo-auto-trainning/training-api/src',
    '/content/yolo-auto-trainning/training-api/src/training',
    '/content/yolo-auto-trainning/training-api/src/api',
    '/content/yolo-auto-trainning/src',
]

for d in dirs_to_check:
    exists = os.path.exists(d)
    print(f"{'✅' if exists else '❌'} {d}")

print("\n" + "="*50)
print("尝试导入核心模块:")

# 尝试导入训练模块
try:
    from training_api.src.training.runner import YOLOTrainer
    print("✅ YOLOTrainer 导入成功")
except Exception as e:
    print(f"❌ YOLOTrainer 导入失败: {e}")

try:
    from training_api.src.training.config import DEFAULT_TRAINING_CONFIG
    print("✅ TrainingConfig 导入成功")
except Exception as e:
    print(f"❌ TrainingConfig 导入失败: {e}")

# 启动 Training API 服务
# 注意: Colab 环境可能不支持长时间运行服务

import os
import sys
import threading
import time

# 重新设置路径 (确保在克隆之后)
sys.path.insert(0, '/content/yolo-auto-trainning')
sys.path.insert(0, '/content/yolo-auto-trainning/training-api/src')
sys.path.insert(0, '/content/yolo-auto-trainning/training-api')

# 设置环境变量
os.environ["INTERNAL_API_KEY"] = "test-api-key-colab"
os.environ["TRAINING_API_URL"] = "http://localhost:8001"

print("准备启动 Training API...")
print("="*50)

# 启动服务
from training_api.src.api.gateway import app
import uvicorn

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info")

# 在后台线程中启动
api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("✅ Training API 已启动: http://localhost:8001")
print("   API 文档: http://localhost:8001/docs")
print("\n测试 API 端点:")

# 等待服务启动
time.sleep(3)

# 测试 API
import httpx

try:
    # 健康检查
    response = httpx.get("http://localhost:8001/health", timeout=5)
    print(f"✅ GET /health: {response.status_code}")
    
    # 测试训练端点 (需要 API Key)
    headers = {"X-API-Key": "test-api-key-colab"}
    
    # 获取训练状态 (无任务)
    response = httpx.get(
        "http://localhost:8001/train/status/test-task",
        headers=headers,
        timeout=5
    )
    print(f"✅ GET /train/status: {response.status_code}")
    
except Exception as e:
    print(f"API 测试完成 (服务可能已停止): {e}")

print("\n注意: Colab 免费版运行时间有限，建议在 Pro/Pro+ 上测试完整 API 功能")